In [1]:
import time



class TimerDecorator:
    def __init__(self, func=None, *, print_each_call=False): #wszystkie argumenty po * muszą być podane jako nazwane
        self.func = func
        self.print_each_call = print_each_call
        self.times = []


#args to argumenty pozycyjne, kwargs to argumenty nazwane
    def __call__(self, *args, **kwargs): #__call__(*args, **kwargs) znaczy że dekorator może być używany z argumentami, rozpakowuje argumenty do wywołania funkcji
        start = time.perf_counter()
        result = self.func(*args, **kwargs)
        end = time.perf_counter()

        duration = end - start
        self.times.append(duration)

        if self.print_each_call:
            print(f"[{self.func.__name__}] czas: {duration:.6f}s")

        return result

    def stats(self):
        if not self.times:
            return {
                "function": self.func.__name__,
                "calls": 0,
                "min": None,
                "max": None,
                "avg": None
            }

        return {
            "function": self.func.__name__,
            "calls": len(self.times),
            "min": min(self.times),
            "max": max(self.times),
            "avg": sum(self.times) / len(self.times)
        }

    def print_stats(self):
        s = self.stats()
        print("staty")
        print(f"Funkcja: {s['function']}")
        print(f"Wywołania: {s['calls']}")
        print(f"Min czas: {s['min']:.6f}s")
        print(f"Max czas: {s['max']:.6f}s")
        print(f"Średni czas: {s['avg']:.6f}s")


# test

@TimerDecorator(print_each_call=True)
def slow_function(n):
    time.sleep(n)
    return n


@TimerDecorator(print_each_call=True)
def add(a, b):
    time.sleep(0.1)
    return a + b


# testy
slow_function(0.2)
slow_function(0.3)
add(2, 3)
add(10, 20)

# statystyki
slow_function.print_stats()
add.print_stats()

TypeError: 'NoneType' object is not callable

In [ ]:



# =========================================================
# ZADANIE 2 — TEMPERATURE
# =========================================================

class Temperature:
    def __init__(self, celsius):
        self._celsius = None
        self.celsius = celsius

    @property
    def celsius(self):
        return self._celsius

    @celsius.setter
    def celsius(self, value):
        self._set_from_celsius(value)

    @property
    def fahrenheit(self):
        return self._celsius * 9 / 5 + 32

    @fahrenheit.setter
    def fahrenheit(self, value):
        c = (value - 32) * 5 / 9
        self._set_from_celsius(c)

    @property
    def kelvin(self):
        return self._celsius + 273.15

    @kelvin.setter
    def kelvin(self, value):
        if value < 0:
            value = 0
        c = value - 273.15
        self._set_from_celsius(c)

    def _set_from_celsius(self, value):
        if value < -273.15:
            value = -273.15
        self._celsius = value

    @property
    def is_freezing(self):
        return self._celsius <= 0

    @property
    def is_boiling(self):
        return self._celsius >= 100

    @property
    def state(self):
        if self._celsius < 0:
            return "solid"
        elif self._celsius < 100:
            return "liquid"
        return "gas"

    @classmethod
    def from_fahrenheit(cls, f):
        return cls((f - 32) * 5 / 9)

    @classmethod
    def from_kelvin(cls, k):
        return cls(k - 273.15 if k >= 0 else -273.15)

    def __repr__(self):
        return f"Temperature({self._celsius:.2f}°C)"


# =========================================================
# ZADANIE 3 — GENERATOR LCG
# =========================================================

def lcg(seed, a, c, m, N, normalized=False):
    x = seed
    for _ in range(N):
        x = (a * x + c) % m
        if normalized:
            yield x / m
        else:
            yield x


# =========================================================
# ZADANIE 4 — WŁASNE PROPERTY (DESKRYPTOR)
# =========================================================

class MyProperty:
    def __init__(self, fget=None, fset=None):
        self.fget = fget
        self.fset = fset

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        return self.fget(obj)

    def __set__(self, obj, value):
        if not self.fset:
            raise AttributeError("Can't set attribute")
        self.fset(obj, value)

    def setter(self, func):
        return MyProperty(self.fget, func)


class Example:
    def __init__(self):
        self._value = 0

    def get_value(self):
        return self._value

    def set_value(self, v):
        if v < 0:
            v = 0
        self._value = v

    value = MyProperty(get_value)
    value = value.setter(set_value)


# =========================================================
# DEMO (URUCHOMIENIE)
# =========================================================

if __name__ == "__main__":

    print("=== ZADANIE 1 ===")

    timer = TimerDecorator(print_each_call=True, max_history=5)

    @timer
    def slow_function():
        time.sleep(0.1)

    @timer
    def compute(n):
        return sum(i * i for i in range(n))

    for _ in range(3):
        slow_function()

    for _ in range(5):
        compute(10000)

    slow_function.print_stats()
    compute.print_stats()

    print("\n=== ZADANIE 2 ===")

    t = Temperature(25)
    print(t)
    print(t.fahrenheit)
    print(t.kelvin)

    t.fahrenheit = 32
    print(t)

    t.kelvin = -100
    print(t.kelvin)

    print("freezing:", t.is_freezing)
    print("state:", t.state)

    print("\n=== ZADANIE 3 ===")

    gen = lcg(1, 5, 3, 16, 5)

    print("next():")
    print(next(gen))
    print(next(gen))

    print("for:")
    for val in lcg(1, 5, 3, 16, 5):
        print(val)

    print("normalized:")
    for val in lcg(1, 5, 3, 16, 5, normalized=True):
        print(val)

    print("\n=== ZADANIE 4 ===")

    e = Example()
    e.value = 10
    print(e.value)

    e.value = -5
    print(e.value)

=== ZADANIE 1 ===
[slow_function] czas: 0.100175s
[slow_function] czas: 0.100184s
[slow_function] czas: 0.100489s
[compute] czas: 0.001510s
[compute] czas: 0.001684s
[compute] czas: 0.001294s
[compute] czas: 0.000996s
[compute] czas: 0.001205s

Statystyki dla slow_function:
Liczba wywołań: 3
Min: 0.100175s
Max: 0.100489s
Avg: 0.100283s

Statystyki dla compute:
Liczba wywołań: 5
Min: 0.000996s
Max: 0.001684s
Avg: 0.001338s

=== ZADANIE 2 ===
Temperature(25.00°C)
77.0
298.15
Temperature(0.00°C)
0.0
freezing: True
state: solid

=== ZADANIE 3 ===
next():
8
11
for:
8
11
10
5
12
normalized:
0.5
0.6875
0.625
0.3125
0.75

=== ZADANIE 4 ===
10
0
